<a href="https://colab.research.google.com/github/osamamohamaed/Myprojects/blob/main/CNN_ViT_CIFAR10_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Classification — CNN & Vision Transformer (ViT)
**Course:** Machine Learning (ANIT-361) | **University of Ha'il**

---
## Objectives
- Build and train a **CNN** model on CIFAR-10
- Build and train a **Vision Transformer (ViT)** model on CIFAR-10
- Compare both models' performance
- Evaluate using accuracy curves, confusion matrix, and classification report

---
## Step 1: Install & Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

---
## Step 2: Load & Explore CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Training: {x_train.shape} | Test: {x_test.shape}')
print(f'Pixel range: [{x_train.min()}, {x_train.max()}]')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train.flatten() == i)[0][0]
    ax.imshow(x_train[idx])
    ax.set_title(class_names[i], fontsize=13, fontweight='bold')
    ax.axis('off')
plt.suptitle('CIFAR-10 — One Sample per Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3: Preprocess

In [ ]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32') / 255.0

y_train_enc = to_categorical(y_train, 10)
y_test_enc  = to_categorical(y_test, 10)

# Integer labels for evaluation
y_true = y_test.flatten()

print(f'Normalized range: [{x_train_norm.min()}, {x_train_norm.max()}]')
print(f'Label example: {y_train[0][0]} -> {y_train_enc[0]}')

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train_norm)

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax in axes:
    ax.imshow(datagen.random_transform(x_train_norm[0]))
    ax.axis('off')
plt.suptitle('Augmented Versions of the Same Image', fontsize=13)
plt.tight_layout()
plt.show()

---
# Part A: CNN Model
---
## Step 4A: Build CNN

In [ ]:
cnn_model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),
    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),
    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),
    # Head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

## Step 5A: Train CNN

In [ ]:
cnn_callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

cnn_history = cnn_model.fit(
    datagen.flow(x_train_norm, y_train_enc, batch_size=64),
    epochs=30,
    validation_data=(x_test_norm, y_test_enc),
    callbacks=cnn_callbacks,
    verbose=1
)

## Step 6A: Evaluate CNN

In [ ]:
cnn_loss, cnn_acc = cnn_model.evaluate(x_test_norm, y_test_enc, verbose=0)
print(f'CNN Test Accuracy: {cnn_acc:.4f} ({cnn_acc*100:.2f}%)')
print(f'CNN Test Loss:     {cnn_loss:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(cnn_history.history['accuracy'],     label='Train', linewidth=2)
ax1.plot(cnn_history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_title('CNN — Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(cnn_history.history['loss'],     label='Train', linewidth=2)
ax2.plot(cnn_history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_title('CNN — Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
cnn_pred = np.argmax(cnn_model.predict(x_test_norm), axis=1)
cm_cnn   = confusion_matrix(y_true, cnn_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('CNN — Confusion Matrix', fontsize=15, fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

print('\nCNN Classification Report:')
print(classification_report(y_true, cnn_pred, target_names=class_names))

---
# Part B: Vision Transformer (ViT) Model
---
## Step 4B: Build ViT from Scratch

In [ ]:
# ── Custom ViT Layers ──────────────────────────────────────────

class PatchExtract(layers.Layer):
    """Split image into fixed-size patches."""
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patch_dim = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dim])

    def get_config(self):
        config = super().get_config()
        config.update({'patch_size': self.patch_size})
        return config


class PatchEmbedding(layers.Layer):
    """Project patches + add positional encoding."""
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = layers.Dense(embed_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=embed_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        config = super().get_config()
        config.update({'num_patches': self.num_patches})
        return config


class TransformerBlock(layers.Layer):
    """Multi-Head Self-Attention + Feed-Forward block."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att      = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn      = models.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dense(embed_dim),
        ])
        self.norm1    = layers.LayerNormalization(epsilon=1e-6)
        self.norm2    = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs):   # No 'training' argument — Keras handles it
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.norm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.norm2(out1 + ffn_output)


print('Custom ViT layers defined successfully!')

In [ ]:
# ── ViT Hyperparameters ────────────────────────────────────────
IMAGE_SIZE             = 32
PATCH_SIZE             = 4
NUM_PATCHES            = (IMAGE_SIZE // PATCH_SIZE) ** 2   # 64
EMBED_DIM              = 64
NUM_HEADS              = 4
FF_DIM                 = 128
NUM_TRANSFORMER_BLOCKS = 4
DROPOUT_RATE           = 0.1

print(f'Patch size:          {PATCH_SIZE}x{PATCH_SIZE}')
print(f'Number of patches:   {NUM_PATCHES}')
print(f'Embedding dimension: {EMBED_DIM}')
print(f'Transformer blocks:  {NUM_TRANSFORMER_BLOCKS}')

In [ ]:
# ── Build ViT ──────────────────────────────────────────────────
def build_vit():
    inputs  = layers.Input(shape=(32, 32, 3))
    patches = PatchExtract(PATCH_SIZE)(inputs)
    encoded = PatchEmbedding(NUM_PATCHES, EMBED_DIM)(patches)

    for _ in range(NUM_TRANSFORMER_BLOCKS):
        encoded = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT_RATE)(encoded)

    encoded = layers.GlobalAveragePooling1D()(encoded)
    encoded = layers.Dropout(0.3)(encoded)
    encoded = layers.Dense(128, activation='gelu')(encoded)
    encoded = layers.Dropout(0.3)(encoded)
    outputs = layers.Dense(10, activation='softmax')(encoded)
    return models.Model(inputs, outputs)


vit_model = build_vit()
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
vit_model.summary()

### Visualize Patches

In [ ]:
sample_img = x_train_norm[0]
fig, axes  = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(sample_img)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(sample_img)
for i in range(0, IMAGE_SIZE + 1, PATCH_SIZE):
    axes[1].axhline(y=i - 0.5, color='red', linewidth=1.5)
    axes[1].axvline(x=i - 0.5, color='red', linewidth=1.5)
axes[1].set_title(f'{NUM_PATCHES} Patches ({PATCH_SIZE}x{PATCH_SIZE} each)', fontsize=13)
axes[1].axis('off')

plt.suptitle('How ViT Splits an Image into Patches', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 5B: Train ViT

In [ ]:
vit_callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

vit_history = vit_model.fit(
    datagen.flow(x_train_norm, y_train_enc, batch_size=64),
    epochs=30,
    validation_data=(x_test_norm, y_test_enc),
    callbacks=vit_callbacks,
    verbose=1
)

## Step 6B: Evaluate ViT

In [ ]:
vit_loss, vit_acc = vit_model.evaluate(x_test_norm, y_test_enc, verbose=0)
print(f'ViT Test Accuracy: {vit_acc:.4f} ({vit_acc*100:.2f}%)')
print(f'ViT Test Loss:     {vit_loss:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(vit_history.history['accuracy'],     label='Train', linewidth=2)
ax1.plot(vit_history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_title('ViT — Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(vit_history.history['loss'],     label='Train', linewidth=2)
ax2.plot(vit_history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_title('ViT — Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
vit_pred = np.argmax(vit_model.predict(x_test_norm), axis=1)
cm_vit   = confusion_matrix(y_true, vit_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names)
plt.title('ViT — Confusion Matrix', fontsize=15, fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

print('\nViT Classification Report:')
print(classification_report(y_true, vit_pred, target_names=class_names))

---
# Part C: Comparison — CNN vs ViT
---

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cnn_history.history['val_accuracy'], label='CNN', linewidth=2, color='#2563EB')
ax1.plot(vit_history.history['val_accuracy'], label='ViT', linewidth=2, color='#8B5CF6')
ax1.set_title('Validation Accuracy — CNN vs ViT', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=12); ax1.grid(True, alpha=0.3)

ax2.plot(cnn_history.history['val_loss'], label='CNN', linewidth=2, color='#2563EB')
ax2.plot(vit_history.history['val_loss'], label='ViT', linewidth=2, color='#8B5CF6')
ax2.set_title('Validation Loss — CNN vs ViT', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=12); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
cnn_per_class = cm_cnn.diagonal() / cm_cnn.sum(axis=1)
vit_per_class = cm_vit.diagonal() / cm_vit.sum(axis=1)

x     = np.arange(len(class_names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, cnn_per_class * 100, width, label='CNN', color='#2563EB', alpha=0.85)
bars2 = ax.bar(x + width/2, vit_per_class * 100, width, label='ViT', color='#8B5CF6', alpha=0.85)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy — CNN vs ViT', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=11)
ax.legend(fontsize=12); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 110)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
print('=' * 50)
print('       FINAL RESULTS COMPARISON')
print('=' * 50)
print(f'  CNN Test Accuracy:  {cnn_acc*100:.2f}%')
print(f'  ViT Test Accuracy:  {vit_acc*100:.2f}%')
print(f'  CNN Test Loss:      {cnn_loss:.4f}')
print(f'  ViT Test Loss:      {vit_loss:.4f}')
print('=' * 50)
winner = 'CNN' if cnn_acc > vit_acc else 'ViT'
diff   = abs(cnn_acc - vit_acc) * 100
print(f'\n  Winner: {winner} (by {diff:.2f}%)')
print(f'\n  CNN Parameters: {cnn_model.count_params():,}')
print(f'  ViT Parameters: {vit_model.count_params():,}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
indices   = np.random.choice(len(x_test_norm), 5, replace=False)

for i, idx in enumerate(indices):
    img  = x_test_norm[idx]
    true = class_names[y_true[idx]]

    cnn_label = class_names[cnn_pred[idx]]
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'True: {true}\nCNN: {cnn_label}', fontsize=10,
                          color='green' if cnn_label == true else 'red', fontweight='bold')
    axes[0, i].axis('off')

    vit_label = class_names[vit_pred[idx]]
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'True: {true}\nViT: {vit_label}', fontsize=10,
                          color='green' if vit_label == true else 'red', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CNN', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('ViT', fontsize=14, fontweight='bold')
plt.suptitle('Sample Predictions (Green = Correct, Red = Wrong)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Conclusion

| Aspect | CNN | ViT |
|--------|-----|-----|
| Feature Extraction | Local (convolution filters) | Global (self-attention) |
| Strengths | Simple, efficient on small images | Flexible, captures long-range dependencies |
| Limitations | Limited receptive field per layer | Needs more data/parameters |

### Possible Improvements
- **Dropout & BatchNorm** for regularization
- **Learning rate scheduling** for better convergence
- **Transfer learning** (pre-trained ResNet or ViT-Base)
- **Ensemble** CNN + ViT predictions for best results